# COT Analysis

#### Install required packages

In [20]:
# !pip install cot-reports

In [ ]:
import cot_reports as cot
import pandas as pd
import numpy as np
import logging
from pathlib import Path

# project root = nearest folder (cwd or a parent) holding config.py, so paths work from docs/ or the root
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
LOG_FILE = PROJECT_ROOT / "logs" / "pipeline.log"
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("cot_pipeline")
logger.setLevel(logging.INFO)
if not logger.handlers:  # avoid duplicate handlers when the cell is re-run
    file_handler = logging.FileHandler(LOG_FILE)
    file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(file_handler)

### Declare important variables for extraction purposes.

In [22]:
DATA_DIR = PROJECT_ROOT / "data"
SIGNAL_DIR = PROJECT_ROOT / "signal"
MARKET_COL = "Market and Exchange Names"
DATE_COL = "As of Date in Form YYYY-MM-DD"

'''
special fields for analysing the commitment of traders report
'''
SPECIAL_COLUMNS = [
    MARKET_COL, DATE_COL, "Noncommercial Positions-Long (All)",
    "Noncommercial Positions-Short (All)" , "Change in Noncommercial-Long (All)", 
    "Change in Noncommercial-Short (All)", "Open Interest (All)", "Change in Open Interest (All)"
]

'''
specific markets and exchanges with varying names  
changes over the years
'''
MARKETS_AND_EXCHANGES = [ 
    "MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE",
    "MEXICAN PESO - INTERNATIONAL MONETARY MARKET",
    "JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE",
    "JAPANESE YEN - INTERNATIONAL MONETARY MARKET",
    "EURO FX - CHICAGO MERCANTILE EXCHANGE",
    "EURO FX - INTERNATIONAL MONETARY",
    "DJIA Consolidated - CHICAGO BOARD OF TRADE",
    "DOW JONES INDUSTRIAL AVERAGE - CHICAGO BOARD OF TRADE",
    "NASDAQ-100 Consolidated - CHICAGO MERCANTILE EXCHANGE",
    "NASDAQ-100 STOCK INDEX - CHICAGO MERCANTILE EXCHANGE",
    "NASDAQ-100 STOCK INDEX - INTERNATIONAL MONETARY MARKET",
    "NEW ZEALAND DOLLAR - CHICAGO MERCANTILE EXCHANGE",
    "NEW ZEALAND DOLLARS - CHICAGO MERCANTILE EXCHANGE",
    "NEW ZEALAND DOLLARS - INTERNATIONAL MONETARY MARKET",
    "NZ DOLLAR - CHICAGO MERCANTILE EXCHANGE",
    "AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE",
    "AUSTRALIAN DOLLARS - CHICAGO MERCANTILE EXCHANGE",
    "AUSTRALIAN DOLLARS - INTERNATIONAL MONETARY MARKET",
    "CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE",
    "CANADIAN DOLLAR - INTERNATIONAL MONETARY MARKET",
    "SWISS FRANC - CHICAGO MERCANTILE EXCHANGE",
    "SWISS FRANC - INTERNATIONAL MONETARY MARKET",
    "BRITISH POUND STERLING - INTERNATIONAL MONETARY MARKET",
    "BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE",
    "POUND STERLING - CHICAGO MERCANTILE EXCHANGE",
    "POUND STERLING - INTERNATIONAL MONETARY MARKET",
    "BRITISH POUND - CHICAGO MERCANTILE EXCHANGE",
    "U.S. DOLLAR INDEX - NEW YORK COTTON EXCHANGE",
    "U.S. DOLLAR INDEX - ICE FUTURES U.S.",
    "U.S. DOLLAR INDEX - NEW YORK BOARD OF TRADE",
    "USD INDEX - ICE FUTURES U.S.",
    "1000 TROY OUNCE SILVER - CHICAGO BOARD OF TRADE",
    "SILVER - CHICAGO BOARD OF TRADE",
    "SILVER, 5000 TROY OZ - CHICAGO BOARD OF TRADE",
    "SILVER - COMMODITY EXCHANGE INC.",
    "COPPER - COMMODITY EXCHANGE INC.",
    "COPPER-GRADE #1 - COMMODITY EXCHANGE INC.",
    "COPPER- #1 - COMMODITY EXCHANGE INC.",
    "GOLD, 100 TROY OZ - CHICAGO BOARD OF TRADE",
    "GOLD - COMMODITY EXCHANGE INC.",
    "GOLD - INTERNATIONAL MONETARY MARKET",
    "PLATINUM - NEW YORK MERCANTILE EXCHANGE",
    "PLATINUM - COMMODITY EXCHANGE INC.",
    "BITCOIN - CHICAGO MERCANTILE EXCHANGE",
    "CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE",
]

'''
naming conventions we will adopt
'''
SYMBOL_NAMES = [
    "MEXICAN PESO",
    "JAPANESE YEN",
    "EURO FX",
    "DJIA Consolidated",
    "NASDAQ-100",
    "NEW ZEALAND",
    "AUSTRALIAN",
    "CANADIAN DOLLAR",
    "SWISS FRANC",
    "POUND",
    "U.S. DOLLAR INDEX",
    "SILVER",
    "COPPER",
    "GOLD",
    "PLATINUM",
    "BITCOIN",
    "CRUDE OIL"
]

'''
paths and column names used when reading/writing symbol files
'''

'\npaths and column names used when reading/writing symbol files\n'

### Get legacy futures from 1986 to 2016 matching the market and exchanges defined above.

In [ ]:
# code to download and save futures data.
# legacy_fut = cot.cot_hist(cot_report_type = "legacy_fut", store_txt=True, verbose=True)

'''
saved legacy fut file to speed up future processing
'''
# load legacy_fut to csv 
legacy_fut = pd.read_csv(PROJECT_ROOT / "old_data" / "FUT86_16.txt", low_memory=False)

In [24]:
legacy_fut.shape

(146677, 129)

In [25]:
legacy_fut.columns

Index(['Market and Exchange Names', 'As of Date in Form YYMMDD',
       'As of Date in Form YYYY-MM-DD', 'CFTC Contract Market Code',
       'CFTC Market Code in Initials', 'CFTC Region Code',
       'CFTC Commodity Code', 'Open Interest (All)',
       'Noncommercial Positions-Long (All)',
       'Noncommercial Positions-Short (All)',
       ...
       'Concentration-Gross LT =8 TDR-Long (Other)',
       'Concentration-Gross LT =8 TDR-Short(Other)',
       'Concentration-Net LT =4 TDR-Long (Other)',
       'Concentration-Net LT =4 TDR-Short (Other)',
       'Concentration-Net LT =8 TDR-Long (Other)',
       'Concentration-Net LT =8 TDR-Short (Other)', 'Contract Units',
       'CFTC Contract Market Code (Quotes)',
       'CFTC Market Code in Initials (Quotes)',
       'CFTC Commodity Code (Quotes)'],
      dtype='object', length=129)

In [26]:
# get number of unique markets and exchanges in the legacy_fut dataset
len(np.array(legacy_fut[MARKET_COL].unique()))

820

In [27]:
def get_special_fut(legacy_fut, markets_and_exchanges, special_columns=SPECIAL_COLUMNS):
    """
    Function to filter the legacy_fut dataset for specific markets and exchanges.
    
    Parameters:
    legacy_fut (DataFrame): The original legacy futures dataset.
    markets_and_exchanges (list): List of market and exchange names to filter by.
    
    Returns:
    DataFrame: Filtered and sorted DataFrame containing only the specified markets and exchanges.
    """
    # from legacy_fut select on the required Market and Exchange names and then sort the data
    special_fut = legacy_fut[legacy_fut[MARKET_COL].isin(markets_and_exchanges)].sort_values(by=MARKET_COL)
    ""
    return special_fut[special_columns]

special_fut = get_special_fut(legacy_fut, MARKETS_AND_EXCHANGES, SPECIAL_COLUMNS)

In [28]:
special_fut.shape

(19472, 8)

In [29]:
len(special_fut[MARKET_COL].unique())

39

In [30]:
print(np.array(special_fut[MARKET_COL].unique()))

['1000 TROY OUNCE SILVER - CHICAGO BOARD OF TRADE'
 'AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE'
 'AUSTRALIAN DOLLARS - CHICAGO MERCANTILE EXCHANGE'
 'AUSTRALIAN DOLLARS - INTERNATIONAL MONETARY MARKET'
 'BRITISH POUND STERLING - CHICAGO MERCANTILE EXCHANGE'
 'BRITISH POUND STERLING - INTERNATIONAL MONETARY MARKET'
 'CANADIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE'
 'CANADIAN DOLLAR - INTERNATIONAL MONETARY MARKET'
 'COPPER - COMMODITY EXCHANGE INC.'
 'COPPER-GRADE #1 - COMMODITY EXCHANGE INC.'
 'CRUDE OIL, LIGHT SWEET-WTI - ICE FUTURES EUROPE'
 'DJIA Consolidated - CHICAGO BOARD OF TRADE'
 'DOW JONES INDUSTRIAL AVERAGE - CHICAGO BOARD OF TRADE'
 'EURO FX - CHICAGO MERCANTILE EXCHANGE' 'GOLD - COMMODITY EXCHANGE INC.'
 'GOLD - INTERNATIONAL MONETARY MARKET'
 'GOLD, 100 TROY OZ - CHICAGO BOARD OF TRADE'
 'JAPANESE YEN - CHICAGO MERCANTILE EXCHANGE'
 'JAPANESE YEN - INTERNATIONAL MONETARY MARKET'
 'MEXICAN PESO - CHICAGO MERCANTILE EXCHANGE'
 'MEXICAN PESO - INTERNATIONAL MONETARY MAR

In [31]:
# Handle unique case for the DJIA, NZ across the entire DataFrame by unifying market and exchange names
replacement_dict = {
        'DOW JONES INDUSTRIAL AVERAGE - CHICAGO BOARD OF TRADE': 'DJIA Consolidated - CHICAGO BOARD OF TRADE',
        'USD INDEX - ICE FUTURES U.S.': 'U.S. DOLLAR INDEX - ICE FUTURES U.S.'
    }

def unify_market_exchange_names(dataframe, replacement_dict):
    """
    Function to unify market and exchange names in the DataFrame based on a replacement dictionary.
    
    Parameters:
    dataframe (DataFrame): The DataFrame containing market and exchange names.
    replacement_dict (dict): Dictionary mapping old names to new unified names.
    
    Returns:
    DataFrame: Updated DataFrame with unified market and exchange names.
    """
    dataframe[MARKET_COL] = dataframe[MARKET_COL].replace(replacement_dict)
    return dataframe

# special_fut = unify_market_exchange_names(special_fut, replacement_dict)

In [32]:
def save_historical_data(symbol_names, special_fut, markets_and_exchanges):
    for sn in symbol_names:
        dfs = []
        results = pd.DataFrame()
            
        # Loop through markets and exchanges and append similar symbols that are similar but with varying names
        # as a result of difference in naming over the years.
        for mx in markets_and_exchanges: 
            if sn in mx:
                dfs.append(special_fut[special_fut[MARKET_COL]==mx])
        
        # Loop through the list and concatenate 
        for df in dfs:
            # Concatenate vertically (along rows)
            results = pd.concat([results, df], axis=0).sort_values(by=DATE_COL, ascending=False)
        # Save COT data (special colunmns) of the different symbols in csv.
        results.to_csv(DATA_DIR / f"{sn}.csv", index=False)

    save_historical_data(SYMBOL_NAMES, special_fut, MARKETS_AND_EXCHANGES)

In [33]:
def load_existing_symbol_data(path):
    '''
    Return the saved data for a symbol, or None if the file is missing/unreadable
    (the caller then creates the file from the new data).
    '''
    if not path.exists():
        logger.warning("%s not found - it will be created from the new data", path)
        return None
    try:
        return pd.read_csv(path)
    except (pd.errors.EmptyDataError, pd.errors.ParserError) as err:
        logger.error("Could not read %s (%s) - it will be rebuilt from the new data", path, err)
        return None


def rows_for_symbol(dataframe, symbol, markets_and_exchanges):
    '''
    Return the rows of dataframe whose market name belongs to symbol.
    '''
    names = [mx for mx in markets_and_exchanges if symbol in mx]
    return dataframe[dataframe[MARKET_COL].isin(names)]


def add_net_positions(dataframe):
    '''
    Return a copy sorted newest-first with Net Positions and Change Net Positions.
    '''
    ordered = dataframe.sort_values(by=DATE_COL, ascending=False).reset_index(drop=True)
    net = ordered["Noncommercial Positions-Long (All)"] - ordered["Noncommercial Positions-Short (All)"]
    return ordered.assign(**{"Net Positions": net, "Change Net Positions": net.diff(-1)})


def merge_symbol_data(old_df, new_df):
    '''
    Combine saved and new rows (old_df may be None), dropping repeated reports.
    '''
    frames = [df[SPECIAL_COLUMNS] for df in (old_df, new_df) if df is not None]
    combined = pd.concat(frames, axis=0)
    combined[DATE_COL] = pd.to_datetime(combined[DATE_COL]).dt.strftime("%Y-%m-%d")
    deduped = combined.drop_duplicates(subset=[MARKET_COL, DATE_COL], keep="last")
    if len(deduped) < len(combined):
        logger.info("Dropped %d duplicate rows already present in the saved data", len(combined) - len(deduped))
    return add_net_positions(deduped)


def modify_old_with_new(markets_and_exchanges, symbols_names, dataframe, data_dir=DATA_DIR):
    '''
    markets_and_exchanges - list of interested markets and exchanges.
    symbols_names - list of symbols corresponding to the markets and exchanges.
    dataframe - pandas dataframe containing markets_and_exchanges data for a particular year.
    data_dir - folder holding one csv per symbol; missing files are created.
    '''
    data_dir.mkdir(parents=True, exist_ok=True)
    for sym in symbols_names:
        path = data_dir / f"{sym}.csv"
        new_df = rows_for_symbol(dataframe, sym, markets_and_exchanges)
        if new_df.empty and not path.exists():
            logger.warning("No rows for %s in this report and no saved file - skipped", sym)
            continue
        old_df = load_existing_symbol_data(path)
        if new_df.empty and old_df is None:
            logger.warning("No rows for %s in this report and %s is unreadable - skipped", sym, path)
            continue
        merged = merge_symbol_data(old_df, new_df)
        try:
            merged.to_csv(path, index=False)
            logger.info("Saved %s: %d new rows, %d total", path, len(new_df), len(merged))
        except OSError:
            logger.exception("Failed to write %s", path)

### Get data for legacy futures from 2017 to present

In [34]:
years = np.arange(2017,2027,1)
years

array([2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026])

### Updating old csv files with new data.

In [35]:

def update_historical_data_for_particular_year(year):
    '''
    year - year we want to carry out update on
    '''
    try:
        legacy_fut_yr = cot.cot_year(year = year, cot_report_type = "legacy_fut", store_txt=False, verbose=False)
    except Exception:  # cot_reports raises untyped network/parsing errors
        logger.exception("Could not download the %s legacy futures report - year skipped", year)
        return
    logger.info("Downloaded %s legacy futures report: %d rows", year, len(legacy_fut_yr))
    legacy_fut_yr = legacy_fut_yr[legacy_fut_yr[MARKET_COL].isin(MARKETS_AND_EXCHANGES)].sort_values(by=MARKET_COL)
    modified_legacy_fut_yr = unify_market_exchange_names(legacy_fut_yr.copy(), replacement_dict)
    modify_old_with_new(MARKETS_AND_EXCHANGES, SYMBOL_NAMES, modified_legacy_fut_yr)

def update_historical_data_across_multiple_years(years):
    for yr in years:
       update_historical_data_for_particular_year(yr)

'''
uncomment line to run code for several years.
'''

update_historical_data_across_multiple_years(years)

### Computing the Open Interest Code for determining price movements for assets

In [36]:
from IPython.display import display, HTML

# Sample Table in HTML
table = """
<table>
  <tr>
    <th>Price</th>
    <th>Open Interest</th>
    <th>Interpretation</th>
    <th>Signal</th>
  </tr>
  <tr>
    <td>Rising</td>
    <td>Rising</td>
    <td>Market is Strong(Bullish)</td>
    <td>1</td>
  </tr>
  <tr>
    <td>Rising</td>
    <td>Falling</td>
    <td>Market is weakening(Bullish Reversal)</td>
    <td>4</td>
  </tr>
  <tr>
    <td>Falling</td>
    <td>Rising</td>
    <td>Market is Weak(Bearish)</td>
    <td>2</td>
  </tr>
  <tr>
    <td>Falling</td>
    <td>Falling</td>
    <td>Market is Strengthing (Bearish Reversal)</td>
    <td>3</td>
  </tr>
</table>
"""

display(HTML(table))

Price,Open Interest,Interpretation,Signal
Rising,Rising,Market is Strong(Bullish),1
Rising,Falling,Market is weakening(Bullish Reversal),4
Falling,Rising,Market is Weak(Bearish),2
Falling,Falling,Market is Strengthing (Bearish Reversal),3


In [37]:
# need to update the price information from Net Positions to actual percentage change of the associated market.
# the question will be what date are we going to look for the change for? we could try Tuesday or Friday and see.

In [38]:
'''
The price of an asset can be deceptive - it doesn't really tell us where it wants to go. 
So modelling the relationship between price and Open interest can be misleading.

Therefore in the above our understanding of price (where price is intented to go) is substituted with the change in Net Positions
We therefore model the relations between Change in Net Position (price) and Open Interest.
'''
def compute_signal_and_interpretation(row):
    if row['Change Net Positions'] > 0 and row['Change in Open Interest (All)'] > 0:
        return 1, 'Bullish'
    elif row['Change Net Positions'] > 0 and row['Change in Open Interest (All)'] < 0:
        return 4, 'Bullish Reversal'
    elif row['Change Net Positions'] < 0 and row['Change in Open Interest (All)'] > 0:
        return 2, 'Bearish'
    elif row['Change Net Positions'] < 0 and row['Change in Open Interest (All)'] < 0:
        return 3, 'Bearish Reversal'
    else:
        return 0, 'No signal'



In [39]:
# Main operations

signal_columns = [
    DATE_COL, "Noncommercial Positions-Long (All)",
    "Noncommercial Positions-Short (All)" , "Change in Noncommercial-Long (All)", 
    "Change in Noncommercial-Short (All)", "Open Interest (All)", "Net Positions", "Interpretation" 
]

def perform_signal(symbols_names=SYMBOL_NAMES):
    SIGNAL_DIR.mkdir(parents=True, exist_ok=True)
    for sn in symbols_names:
        data_path = DATA_DIR / f"{sn}.csv"
        if not data_path.exists():
            logger.warning("%s not found - no signal file written for %s", data_path, sn)
            continue
        asset = pd.read_csv(data_path)
        
        # Replace non-numeric values with NaN, then convert to int
        asset.loc[:, 'Change in Open Interest (All)'] = (
            pd.to_numeric(asset['Change in Open Interest (All)'], errors='coerce')
            .fillna(0)
            .astype(int)
        )
    
        # Apply the function 2 to each row of the DataFrame and create 'signal' and 'Interpretation' columns
        asset[['signal', 'Interpretation']] = asset.apply(
            compute_signal_and_interpretation,
            axis=1,
            result_type="expand"
        )
        
        # Save the result to a CSV file
        signal_path = SIGNAL_DIR / f"{sn}.csv"
        asset[signal_columns].to_csv(signal_path, index=False)
        logger.info("Saved %s: %d rows", signal_path, len(asset))

# save signal information with/without closing prices.
perform_signal()

### Deductions
- If change in open interest rises so does the change in net position and vice versa
- If the change in net positions is very negative or very positive then it can be an indication that the net positions will revert (rise or drop) soon.
- It is possible that the change in open interest can be a leading indicator for the change in net position (but more investigations needed).

## The End

**References**

- [Learn more on the COT Report](https://www.investopedia.com/terms/c/cot.asp)
- [COT_Reports package by NDelventhal](https://github.com/NDelventhal/cot_reports)